## build_dim_geo
Rebuilds `gold.dim_geo` by carrying `silver.dim_geo` **verbatim** — the same `geo_key` values (plain BIGINT, NOT regenerated identity), so every Gold fact's `geo_key` still resolves (design §2.1). Full rebuild via `INSERT OVERWRITE` (design §2.2), which preserves the G0 table definition, PK/FK constraints, and column COMMENTs. StepLog + transform_detail_log. Spec: `gold_layer_design.md` §3.1.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
# notebook_init injects: SILVER, GOLD, AUDIT, PIPELINE_RUN_ID, STATUS_*, StepLog, Utils,
# transform_detail_log_insert, spark, dbutils, F, datetime, timezone.

STEP_SEQUENCE = 1                       # position owned by the orchestrator (G4)
SOURCE_TABLE  = f"{SILVER}.dim_geo"
TARGET_TABLE  = f"{GOLD}.dim_geo"

# Carried verbatim, IN G0 COLUMN ORDER (gold_ddl.py). geo_key first; audit ts set fresh at write.
CARRY_COLS = [
    "geo_key", "cbsa_code", "cbsa_title", "cbsa_type", "zillow_region_id",
    "primary_state", "state_list", "household_rank", "census_region", "cbsa_population",
]

In [ ]:
# Open the pipeline_step_log row (RUNNING); closed by step.succeed() in the write cell.
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "gold",
    target_table    = TARGET_TABLE,
)
print(f"build_dim_geo: step_log_id={step.step_log_id}")

In [ ]:
# Carry Silver dim_geo verbatim (same key values); set fresh Gold audit timestamps.
try:
    src = spark.table(SOURCE_TABLE)
    rows_read = src.count()
    staged = src.select(
        *[F.col(col_name) for col_name in CARRY_COLS],
        F.current_timestamp().alias("inserted_ts"),
        F.current_timestamp().alias("updated_ts"),
    )
    staged.createOrReplaceTempView("gold_dim_geo_staging")
    step.rows_read = rows_read
    print(f"build_dim_geo: read {rows_read:,} Silver dim_geo rows")
except Exception as e:
    step.fail(e); raise

In [ ]:
# Full rebuild via INSERT OVERWRITE (design §2.2) — preserves the G0 table definition,
# PK/FK constraints, and COMMENTs (saveAsTable(overwrite) would drop them). Staging view
# columns are in G0 order, so the positional INSERT is correct.
transform_started = datetime.now(timezone.utc)
try:
    spark.sql(f"INSERT OVERWRITE TABLE {TARGET_TABLE} SELECT * FROM gold_dim_geo_staging")

    post_count = spark.table(TARGET_TABLE).count()
    if post_count != step.rows_read:
        raise AssertionError(
            f"[{TARGET_TABLE}] Row-count mismatch: carried {step.rows_read:,} from "
            f"{SOURCE_TABLE}, table now has {post_count:,}."
        )
    step.rows_written = post_count
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_SUCCEEDED, started_timestamp=transform_started, rows_read=step.rows_read,
        rows_written=post_count, rows_inserted=post_count, ended_timestamp=datetime.now(timezone.utc))
    step.succeed()
    print(f"build_dim_geo: wrote {post_count:,} rows to {TARGET_TABLE}")
except Exception as e:
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_FAILED, started_timestamp=transform_started, rows_read=step.rows_read,
        error_message=f"{type(e).__name__}: {e}", ended_timestamp=datetime.now(timezone.utc))
    step.fail(e); raise